# Fast reliable court-consideration text-to-JSON enrichment

This notebook is designed for Kaggle T4 runtimes and Qwen3-8B-AWQ.

Key fixes included:

- Uses the local Kaggle model path first when available.
- Removes FlashInfer after vLLM install to avoid the `cannot find -lcuda` JIT/linker crash.
- Tries vLLM with an explicit Triton attention config when supported by the installed vLLM version.
- Falls back automatically to Transformers + GPTQModel AWQ if vLLM cannot start.
- Uses deterministic JSON-only generation, parsing, normalization, and JSONL/CSV outputs.

Run the install cell, then restart the Kaggle session once if packages were changed.

In [ ]:
# Cell 1 - Install dependencies and remove FlashInfer
# Run this once. If Kaggle changes packages, restart the session after this cell.

import sys, subprocess, os

def sh(cmd, check=False):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)

# Core packages for both vLLM and Transformers fallback.
sh(f"{sys.executable} -m pip install -q -U --no-cache-dir pandas tqdm transformers accelerate safetensors gptqmodel")

# vLLM is preferred for speed. If install fails, the notebook still has a Transformers fallback.
sh(f"{sys.executable} -m pip install -q -U --no-cache-dir vllm", check=False)

# Critical Kaggle fix: FlashInfer JIT often fails with '/usr/bin/ld: cannot find -lcuda'.
# Removing it forces vLLM to choose a non-FlashInfer backend where possible.
sh(f"{sys.executable} -m pip uninstall -y flashinfer flashinfer-python flashinfer-python-cu12", check=False)

print("Install cell finished. If any packages were installed/removed, restart the Kaggle session before continuing.")

In [ ]:
# Cell 2 - Imports, environment, and configuration

from __future__ import annotations

import gc
import json
import os
import re
import time
import traceback
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional

# Set before importing vLLM. Some vLLM versions ignore this, but older ones honor it.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7")
os.environ.setdefault("VLLM_ATTENTION_BACKEND", "TRITON_ATTN")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer

print("Python imports OK")
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU {i}: {p.name} total={total/1024**3:.2f} GiB free={free/1024**3:.2f} GiB")

LOCAL_QWEN3_AWQ = Path("/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1")
HF_QWEN3_AWQ = "Qwen/Qwen3-8B-AWQ"

def first_existing_model() -> str:
    return str(LOCAL_QWEN3_AWQ) if LOCAL_QWEN3_AWQ.exists() else HF_QWEN3_AWQ

@dataclass
class Config:
    input_csv: str = "/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv"
    fallback_input_csv: str = "court_consideration.csv"
    output_dir: str = "/kaggle/working"
    output_jsonl: str = "enriched_court_citations_10.jsonl"
    output_preview_csv: str = "enriched_court_citations_10_preview.csv"

    n_rows: int = 10
    random_seed: Optional[int] = None
    sample_random: bool = False
    text_chars: int = 3200

    # Prefer local Kaggle model input. If unavailable, this falls back to Hugging Face.
    model_name: str = first_existing_model()

    # Engine selection: "auto", "vllm", or "transformers".
    # Use "auto" for normal runs. Use "transformers" if vLLM keeps failing on a Kaggle image.
    engine: str = "auto"

    # vLLM-safe defaults for T4. You can increase later after a successful 10/100-row test.
    tensor_parallel_size: int = 1
    gpu_memory_utilization: float = 0.62
    max_model_len: int = 3072
    enforce_eager: bool = True
    max_num_seqs: int = 32
    quantization: str = "awq"
    disable_custom_all_reduce: bool = True
    force_triton_attention: bool = True

    batch_size: int = 8
    max_new_tokens: int = 384
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.02
    enable_thinking: bool = False

cfg = Config()
print(asdict(cfg))

In [ ]:
# Cell 3 - Resolve input and select rows

def resolve_input_path(cfg: Config) -> Path:
    candidates = [
        Path(cfg.input_csv),
        Path(cfg.fallback_input_csv),
        Path("/kaggle/working") / cfg.fallback_input_csv,
        Path("/mnt/data") / cfg.fallback_input_csv,
    ]
    for p in candidates:
        if p.exists():
            return p

    if Path("/kaggle/input").exists():
        patterns = ["**/court_considerations.csv", "**/court_consideration.csv"]
        for pat in patterns:
            found = sorted(Path("/kaggle/input").glob(pat))
            if found:
                return found[0]

    raise FileNotFoundError("Could not find court_considerations.csv. Update cfg.input_csv.")

input_path = resolve_input_path(cfg)
print("Using input:", input_path)

df = pd.read_csv(input_path)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(3)

In [ ]:
# Cell 4 - Infer citation/text columns and sample rows

def pick_column(columns: List[str], preferred: List[str], contains_any: List[str]) -> Optional[str]:
    lower_map = {c.lower(): c for c in columns}
    for name in preferred:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    for c in columns:
        lc = c.lower()
        if any(token in lc for token in contains_any):
            return c
    return None

columns = list(df.columns)

citation_col = pick_column(
    columns,
    preferred=["citation", "cite", "court_citation", "authority_citation", "consideration_citation"],
    contains_any=["citation", "cite", "bge"],
)

text_col = pick_column(
    columns,
    preferred=["text", "consideration_text", "paragraph_text", "content", "raw_text", "body"],
    contains_any=["text", "content", "paragraph", "consideration", "body"],
)

if citation_col is None:
    raise ValueError("Could not infer citation column. Set citation_col manually.")
if text_col is None:
    raise ValueError("Could not infer text column. Set text_col manually.")

print("citation_col:", citation_col)
print("text_col:", text_col)

valid = df[df[citation_col].notna() & df[text_col].notna()].copy()
valid[text_col] = valid[text_col].astype(str)
valid = valid[valid[text_col].str.strip().str.len() > 30].copy()

if cfg.sample_random:
    work_df = valid.sample(n=min(cfg.n_rows, len(valid)), random_state=cfg.random_seed)
else:
    work_df = valid.head(cfg.n_rows)

work_df = work_df.reset_index(drop=False).rename(columns={"index": "_source_row"})
print("Selected rows:", len(work_df))
work_df[[citation_col, text_col]].head(10)

In [ ]:
# Cell 5 - Schema, prompt, parsing, and validation helpers

REQUIRED_FIELDS = [
    "legal_area",
    "legal_domain_path",
    "topic",
    "subtopic",
    "micro_topic",
    "concepts_en",
    "terms_original",
    "statute_anchors",
    "case_anchors",
    "doctrinal_rule",
    "legal_test",
    "fact_pattern_tags",
    "procedural_context",
    "paragraph_role",
    "authority_role",
    "outcome_signal",
    "query_phrases_en",
    "summary_en",
]

ROLE_VALUES = {"holding", "reasoning", "facts", "procedural_history", "citation", "dissent", "neutral"}
OUTCOME_VALUES = {"granted", "dismissed", "inadmissible", "remitted", "partial", "none"}
AUTHORITY_VALUES = {
    "leading_decision", "settled_rule", "application", "distinguishing", "procedural", "background", "none"
}

JSON_SCHEMA_TEXT = """
{
  "legal_area": "string",
  "legal_domain_path": ["2-6 short English taxonomy labels"],
  "topic": "string",
  "subtopic": "string",
  "micro_topic": "string",
  "concepts_en": ["4-14 concise English legal concepts"],
  "terms_original": ["4-18 exact legal terms from the original language when present"],
  "statute_anchors": ["statutes/articles explicitly present only"],
  "case_anchors": ["case citations explicitly present only"],
  "doctrinal_rule": "English legal rule supported by the text, or empty string",
  "legal_test": "test/standard supported by the text, or empty string",
  "fact_pattern_tags": ["short factual tags"],
  "procedural_context": "string",
  "paragraph_role": "holding|reasoning|facts|procedural_history|citation|dissent|neutral",
  "authority_role": ["leading_decision|settled_rule|application|distinguishing|procedural|background|none"],
  "outcome_signal": "granted|dismissed|inadmissible|remitted|partial|none",
  "query_phrases_en": ["3-8 search queries a lawyer might ask"],
  "summary_en": "one concise English summary"
}
""".strip()

SYSTEM_PROMPT = (
    "You are a deterministic Swiss legal text-to-JSON enrichment engine. "
    "Return exactly one valid JSON object and nothing else. No markdown. No explanation. "
    "Use English for classification fields. Use only facts, statutes, and case citations supported by the input text. "
    "Do not invent article numbers, laws, case citations, or procedural outcomes."
)

USER_TEMPLATE = """
Citation: {citation}

Original court consideration text:
{text}

Create exactly one JSON object matching this schema:
{schema}

Rules:
- Return JSON only.
- All required keys must be present.
- statute_anchors and case_anchors must contain only anchors explicitly visible in the citation/text.
- If a field is unsupported, use an empty string, empty list, or "none".
""".strip()

def make_messages(citation: str, text: str) -> List[Dict[str, str]]:
    text = (text or "")[: cfg.text_chars]
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(citation=citation, text=text, schema=JSON_SCHEMA_TEXT)},
    ]

def extract_json_object(s: str) -> Dict[str, Any]:
    if not isinstance(s, str):
        raise ValueError("Model output is not a string")
    s = s.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.DOTALL | re.IGNORECASE).strip()

    try:
        obj = json.loads(s)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    start = s.find("{")
    if start < 0:
        raise ValueError(f"No JSON object found: {s[:300]}")

    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(s[start:i+1])

    raise ValueError(f"No balanced JSON object found: {s[:300]}")

def clean_str(x: Any, max_chars: int = 1200) -> str:
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()[:max_chars]

def clean_list(x: Any, max_items: int = 12, max_chars: int = 140) -> List[str]:
    if x is None:
        return []
    if isinstance(x, str):
        x = [x]
    if not isinstance(x, list):
        return []
    out, seen = [], set()
    for item in x:
        s = clean_str(item, max_chars=max_chars)
        if not s:
            continue
        key = s.lower()
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
        if len(out) >= max_items:
            break
    return out

def normalize_enrichment(obj: Dict[str, Any]) -> Dict[str, Any]:
    out = {}
    for field in ["legal_area", "topic", "subtopic", "micro_topic", "doctrinal_rule", "legal_test", "procedural_context", "summary_en"]:
        out[field] = clean_str(obj.get(field, ""))

    out["legal_domain_path"] = clean_list(obj.get("legal_domain_path", []), 6)
    out["concepts_en"] = clean_list(obj.get("concepts_en", []), 14)
    out["terms_original"] = clean_list(obj.get("terms_original", []), 18)
    out["statute_anchors"] = clean_list(obj.get("statute_anchors", []), 10)
    out["case_anchors"] = clean_list(obj.get("case_anchors", []), 10)
    out["fact_pattern_tags"] = clean_list(obj.get("fact_pattern_tags", []), 10)
    out["query_phrases_en"] = clean_list(obj.get("query_phrases_en", []), 8, max_chars=220)

    role = clean_str(obj.get("paragraph_role", "neutral")).lower()
    out["paragraph_role"] = role if role in ROLE_VALUES else "neutral"

    outcome = clean_str(obj.get("outcome_signal", "none")).lower()
    out["outcome_signal"] = outcome if outcome in OUTCOME_VALUES else "none"

    auth = [x.lower() for x in clean_list(obj.get("authority_role", []), 7)]
    auth = [x for x in auth if x in AUTHORITY_VALUES]
    out["authority_role"] = auth or ["none"]

    for k in REQUIRED_FIELDS:
        out.setdefault(k, [] if k in {"legal_domain_path", "concepts_en", "terms_original", "statute_anchors", "case_anchors", "fact_pattern_tags", "authority_role", "query_phrases_en"} else "")

    return out

In [ ]:
# Cell 6 - Engine loader: vLLM first, Transformers AWQ fallback

class JsonEngine:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.kind = None
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
        self.llm = None
        self.model = None

        engine = cfg.engine.lower().strip()
        if engine not in {"auto", "vllm", "transformers"}:
            raise ValueError("cfg.engine must be 'auto', 'vllm', or 'transformers'")

        if engine in {"auto", "vllm"}:
            try:
                self._load_vllm()
                self.kind = "vllm"
                print("Engine: vLLM")
                return
            except Exception as exc:
                print("[vLLM] load failed. Root cause follows:")
                print(repr(exc))
                traceback.print_exc(limit=5)
                self._cleanup_cuda()
                if engine == "vllm":
                    raise
                print("[fallback] Switching to Transformers + GPTQModel AWQ.")

        self._load_transformers()
        self.kind = "transformers"
        print("Engine: Transformers")

    def _cleanup_cuda(self):
        try:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
        except Exception:
            pass

    def _attention_config(self):
        if not self.cfg.force_triton_attention:
            return None
        # vLLM >=0.19 exposes AttentionConfig with backend support.
        for import_path in ["vllm.config", "vllm.config.attention"]:
            try:
                mod = __import__(import_path, fromlist=["AttentionConfig"])
                AttentionConfig = getattr(mod, "AttentionConfig")
                try:
                    return AttentionConfig(backend="TRITON_ATTN")
                except Exception:
                    return AttentionConfig(backend="triton_attn")
            except Exception:
                continue
        return None

    def _load_vllm(self):
        from vllm import LLM, SamplingParams
        self.SamplingParams = SamplingParams

        kwargs = dict(
            model=self.cfg.model_name,
            tensor_parallel_size=self.cfg.tensor_parallel_size,
            gpu_memory_utilization=self.cfg.gpu_memory_utilization,
            max_model_len=self.cfg.max_model_len,
            enforce_eager=self.cfg.enforce_eager,
            trust_remote_code=True,
            disable_custom_all_reduce=self.cfg.disable_custom_all_reduce,
            max_num_seqs=self.cfg.max_num_seqs,
            quantization=self.cfg.quantization,
            disable_log_stats=True,
        )

        attn_cfg = self._attention_config()
        if attn_cfg is not None:
            kwargs["attention_config"] = attn_cfg
            print("[vLLM] using explicit AttentionConfig backend=TRITON_ATTN")
        else:
            print("[vLLM] AttentionConfig unavailable; relying on FlashInfer uninstall + backend auto-selection")

        try:
            self.llm = LLM(**kwargs)
        except TypeError as exc:
            # Some vLLM versions may not accept attention_config through LLM(...).
            if "attention_config" in kwargs:
                print("[vLLM] attention_config rejected; retrying without it after FlashInfer uninstall.")
                kwargs.pop("attention_config", None)
                self.llm = LLM(**kwargs)
            else:
                raise exc

    def _load_transformers(self):
        from transformers import AutoModelForCausalLM
        print("[Transformers] loading tokenizer/model:", self.cfg.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.cfg.model_name,
            device_map="auto",
            dtype=torch.float16,
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        ).eval()

    def _render_prompts(self, rows: List[Dict[str, str]]) -> List[str]:
        prompts = []
        for r in rows:
            messages = make_messages(r["citation"], r["text"])
            try:
                prompt = self.tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=self.cfg.enable_thinking,
                )
            except TypeError:
                prompt = self.tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
            prompts.append(prompt)
        return prompts

    def generate(self, rows: List[Dict[str, str]]) -> List[str]:
        prompts = self._render_prompts(rows)
        if self.kind == "vllm":
            params = self.SamplingParams(
                temperature=self.cfg.temperature,
                top_p=self.cfg.top_p,
                max_tokens=self.cfg.max_new_tokens,
                repetition_penalty=self.cfg.repetition_penalty,
                stop=["<|im_end|>", "</s>"],
            )
            outs = self.llm.generate(prompts, sampling_params=params, use_tqdm=False)
            return [o.outputs[0].text for o in outs]

        # Transformers fallback.
        inputs = self.tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=self.cfg.max_model_len).to(self.model.device)
        with torch.inference_mode():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.cfg.max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                repetition_penalty=self.cfg.repetition_penalty,
                use_cache=True,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        gen = outputs[:, inputs["input_ids"].shape[1]:]
        return self.tokenizer.batch_decode(gen, skip_special_tokens=True)

engine = JsonEngine(cfg)

In [ ]:
# Cell 7 - Run enrichment and write JSONL + preview CSV

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
out_jsonl = out_dir / cfg.output_jsonl
out_csv = out_dir / cfg.output_preview_csv

records = []
failures = []
t0 = time.time()

for start in tqdm(range(0, len(work_df), cfg.batch_size), desc="enrich", unit="batch"):
    batch_df = work_df.iloc[start:start + cfg.batch_size]
    rows = []
    for _, row in batch_df.iterrows():
        rows.append({
            "_source_row": int(row["_source_row"]),
            "citation": str(row[citation_col]),
            "text": str(row[text_col]),
        })

    raws = engine.generate(rows)
    for r, raw in zip(rows, raws):
        base = {
            "_source_row": r["_source_row"],
            "citation": r["citation"],
            "text": r["text"],
        }
        try:
            obj = extract_json_object(raw)
            enrich = normalize_enrichment(obj)
            rec = {**base, **enrich}
            records.append(rec)
        except Exception as exc:
            failures.append({**base, "error": repr(exc), "raw_output": raw})

elapsed = time.time() - t0
print(f"Generated {len(records)} OK, {len(failures)} failed in {elapsed:.1f}s")
if len(work_df):
    print(f"Rows/s: {len(work_df)/max(elapsed, 1e-9):.3f}")

with out_jsonl.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

preview_df = pd.DataFrame(records)
preview_df.to_csv(out_csv, index=False)

if failures:
    fail_path = out_dir / (Path(cfg.output_jsonl).stem + "_failures.jsonl")
    with fail_path.open("w", encoding="utf-8") as f:
        for rec in failures:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print("Failures written to:", fail_path)

print("JSONL:", out_jsonl)
print("Preview CSV:", out_csv)
preview_df.head(10)

In [ ]:
# Cell 8 - Inspect one output

if records:
    print(json.dumps(records[0], ensure_ascii=False, indent=2)[:4000])
else:
    print("No successful records. Check failures.")
    if failures:
        print(json.dumps(failures[0], ensure_ascii=False, indent=2)[:4000])